In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [2]:
raw = spark.read.csv("hdfs://namenode:9000/data/raw/accidentes/", header=True, inferSchema=True)
raw.write.mode("overwrite").parquet("hdfs://namenode:9000/data/staging/accidentes")

In [3]:
raw.count()

7728394

In [3]:
stg = spark.read.parquet("hdfs://namenode:9000/data/staging/accidentes/")

In [4]:
stg.count()

7728394

In [6]:
stg.columns

['ID',
 'Source',
 'Severity',
 'Start_Time',
 'End_Time',
 'Start_Lat',
 'Start_Lng',
 'End_Lat',
 'End_Lng',
 'Distance(mi)',
 'Description',
 'Street',
 'City',
 'County',
 'State',
 'Zipcode',
 'Country',
 'Timezone',
 'Airport_Code',
 'Weather_Timestamp',
 'Temperature(F)',
 'Wind_Chill(F)',
 'Humidity(%)',
 'Pressure(in)',
 'Visibility(mi)',
 'Wind_Direction',
 'Wind_Speed(mph)',
 'Precipitation(in)',
 'Weather_Condition',
 'Amenity',
 'Bump',
 'Crossing',
 'Give_Way',
 'Junction',
 'No_Exit',
 'Railway',
 'Roundabout',
 'Station',
 'Stop',
 'Traffic_Calming',
 'Traffic_Signal',
 'Turning_Loop',
 'Sunrise_Sunset',
 'Civil_Twilight',
 'Nautical_Twilight',
 'Astronomical_Twilight']

In [7]:
from pyspark.sql import functions as F

df_counts = stg.groupBy("ID").count().filter("count > 1")
print("Cantidad de IDs que tienen duplicados:", df_counts.count())

df_counts.select("ID").show(10, truncate=False)

Cantidad de IDs que tienen duplicados: 0
+---+
|ID |
+---+
+---+



In [8]:
from pyspark.sql import functions as F

total_filas = stg.count()
columnas = ["End_Lat", "End_Lng"]

for col_nombre in columnas:
    nulos = stg.filter(F.col(col_nombre).isNull()).count()
    porcentaje = (nulos / total_filas) * 100
    print(f"Columna: {col_nombre} | Vacíos/Nulos: {nulos} | Porcentaje: {porcentaje:.2f}%")

Columna: End_Lat | Vacíos/Nulos: 3402762 | Porcentaje: 44.03%
Columna: End_Lng | Vacíos/Nulos: 3402762 | Porcentaje: 44.03%


In [ ]:
excluir = ["End_Lat", "End_Lng"]

cols_accidentes = [c for c in (stg.columns[0:19] + stg.columns[29:]) if c not in excluir]

accidents = stg.select(cols_accidentes)
accidents.show(2)

+---------+-------+--------+-------------------+-------------------+---------+-----------+------------+--------------------+---------+-----------+---------+-----+----------+-------+----------+------------+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
|       ID| Source|Severity|         Start_Time|           End_Time|Start_Lat|  Start_Lng|Distance(mi)|         Description|   Street|       City|   County|State|   Zipcode|Country|  Timezone|Airport_Code|Crossing|Give_Way|Junction|No_Exit|Railway|Roundabout|Station| Stop|Traffic_Calming|Traffic_Signal|Turning_Loop|Sunrise_Sunset|Civil_Twilight|Nautical_Twilight|Astronomical_Twilight|
+---------+-------+--------+-------------------+-------------------+---------+-----------+------------+--------------------+---------+-----------+---------+-----+----------+-------+----------+------------+--------+--------+-

In [ ]:
weather = stg.select("ID",*stg.columns[19:29])

weather.show(2)

+---------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+--------------------+-------+
|       ID|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|   Weather_Condition|Amenity|
+---------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+--------------------+-------+
|A-5199312|2022-10-07 23:54:00|          79.0|         79.0|       34.0|        29.5|           8.0|             E|            5.0|              0.0|                Fair|  false|
|A-5199313|2023-01-28 03:15:00|          32.0|         19.0|       64.0|       26.21|          10.0|             W|           23.0|              0.0|Partly Cloudy / W...|  false|
+---------+-------------------+--------------+-------------+-----------+------------+--------------+-----

In [11]:
from pyspark.sql import functions as F

cols = [c for c in accidents.columns if c != "ID"]

nulos_por_id = (
    accidents.select(
        "ID",
        F.array_compact(
            F.array(*[F.when(F.col(f"`{c}`").isNull(), F.lit(c)) for c in cols])
        ).alias("columnas_nulas"),
    )
    .withColumn("n_nulos", F.size("columnas_nulas"))
    .filter("n_nulos > 0")
)

nulos_por_id.orderBy(F.desc("n_nulos")).show(10, truncate=False)

+---------+-----------------------------------------------------------------------------------------------------------------+-------+
|ID       |columnas_nulas                                                                                                   |n_nulos|
+---------+-----------------------------------------------------------------------------------------------------------------+-------+
|A-5336388|[City, Zipcode, Timezone, Airport_Code, Sunrise_Sunset, Civil_Twilight, Nautical_Twilight, Astronomical_Twilight]|8      |
|A-3976250|[City, Zipcode, Timezone, Airport_Code, Sunrise_Sunset, Civil_Twilight, Nautical_Twilight, Astronomical_Twilight]|8      |
|A-5183586|[Zipcode, Timezone, Airport_Code, Sunrise_Sunset, Civil_Twilight, Nautical_Twilight, Astronomical_Twilight]      |7      |
|A-5714748|[Zipcode, Timezone, Airport_Code, Sunrise_Sunset, Civil_Twilight, Nautical_Twilight, Astronomical_Twilight]      |7      |
|A-3762491|[Zipcode, Timezone, Airport_Code, Sunrise_Sunset, C

In [12]:
from pyspark.sql import functions as F

total = accidents.count()
us_count = accidents.filter(F.col("Country") == "US").count()
print(f"Porcentaje de registros con 'US': {(us_count / total) * 100:.2f}%")

Porcentaje de registros con 'US': 100.00%


In [7]:
accidents = accidents.drop("Country")


In [8]:
total_antes = accidents.count()

accidents = accidents.dropna()

total_despues = accidents.count()
total_eliminados = total_antes - total_despues

print(f"Registros originales: {total_antes:,}")
print(f"Registros limpios finales en accidents: {total_despues:,}")
print(f"Total de filas eliminadas: {total_eliminados:,}")

Registros originales: 7,728,394
Registros limpios finales en accidents: 7,674,414
Total de filas eliminadas: 53,980


In [24]:
from pyspark.sql import functions as F

cols = [c for c in accidents.columns if c != "ID"]

nulos_por_id = (
    accidents.select(
        "ID",
        F.array_compact(
            F.array(*[F.when(F.col(f"`{c}`").isNull(), F.lit(c)) for c in cols])
        ).alias("columnas_nulas"),
    )
    .withColumn("n_nulos", F.size("columnas_nulas"))
    .filter("n_nulos > 0")
)

nulos_por_id.orderBy(F.desc("n_nulos")).show(10, truncate=False)

+---+--------------+-------+
|ID |columnas_nulas|n_nulos|
+---+--------------+-------+
+---+--------------+-------+



In [21]:
from pyspark.sql import functions as F

total_registros = accidents.count()
print(f"Total de registros: {total_registros:,}\n")
print(f"{'Columna':<40} | {'Faltantes / Vacíos'}")
print("-" * 65)

for col_nombre, dtype in accidents.dtypes:
    if dtype == "string":
        faltantes = accidents.filter(
            F.col(f"`{col_nombre}`").isNull() | 
            (F.col(f"`{col_nombre}`") == "") | 
            (F.col(f"`{col_nombre}`") == " ")
        ).count()
    else:
        faltantes = accidents.filter(F.col(f"`{col_nombre}`").isNull()).count()
        
    print(f"{col_nombre:<40} | {faltantes:,}")

Total de registros: 7,674,414

Columna                                  | Faltantes / Vacíos
-----------------------------------------------------------------
ID                                       | 0
Source                                   | 0
Severity                                 | 0
Start_Time                               | 0
End_Time                                 | 0
Start_Lat                                | 0
Start_Lng                                | 0
Distance(mi)                             | 0
Description                              | 0
Street                                   | 0
City                                     | 0
County                                   | 0
State                                    | 0
Zipcode                                  | 0
Timezone                                 | 0
Airport_Code                             | 0
Crossing                                 | 0
Give_Way                                 | 0
Junction                       

In [ ]:
accidents.printSchema()


root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Crossing: boolean (nullable = true)
 |-- Give_Way: boolean (nullable = true)
 |-- Junction: boolean (nullable = true)
 |-- No_Exit: boolean (nullable = true)
 |-- Railway: boolean (nullable = true)
 |-- Roundabout: boolean (nullable = true)
 |-- Station: boolean (nullable = true)
 |-- Stop: boolean (nullable = true)
 |-- Traffic_Calming: boolean 

In [25]:
from pyspark.sql import functions as F


distancias_anomalas = accidents.filter(F.col("Distance(mi)") < 0).count()
print(f"Registros con distancia negativa (< 0): {distancias_anomalas:,}")


Registros con distancia negativa (< 0): 0


In [9]:
accidents.drop("Zipcode")
accidents.drop("Airport_Code")

DataFrame[ID: string, Source: string, Severity: int, Start_Time: timestamp, End_Time: timestamp, Start_Lat: double, Start_Lng: double, Distance(mi): double, Description: string, Street: string, City: string, County: string, State: string, Zipcode: string, Timezone: string, Crossing: boolean, Give_Way: boolean, Junction: boolean, No_Exit: boolean, Railway: boolean, Roundabout: boolean, Station: boolean, Stop: boolean, Traffic_Calming: boolean, Traffic_Signal: boolean, Turning_Loop: boolean, Sunrise_Sunset: string, Civil_Twilight: string, Nautical_Twilight: string, Astronomical_Twilight: string]

In [51]:

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType
exprs = []
total_filas = accidents.count()
cols = [c for c in accidents.columns if c != "ID"]
tipos = dict(accidents.dtypes)
valores_nulos_texto = ["", "NaN", "nan", "None", "N/A", "NULL"]
for col_nombre in cols:
    if tipos[col_nombre] in ("double", "float"):
        condicion = F.isnan(F.col(col_nombre))
    else: 
        condicion = F.trim(F.col(col_nombre)).isin(valores_nulos_texto)
    exprs.append(F.sum(F.when(condicion, 1).otherwise(0)).alias(col_nombre))
conteos = accidents.agg(*exprs).collect()[0].asDict()

In [52]:
for c in cols:
    n = conteos[c] or 0
    print(f"Columna: {c} | Nulos/NaN: {n} | Porcentaje: {n / total_filas * 100:.2f}%")

Columna: Source | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Severity | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Start_Time | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: End_Time | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Start_Lat | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Start_Lng | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Distance(mi) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Description | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Street | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: City | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: County | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: State | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Zipcode | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Timezone | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Airport_Code | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Crossing | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Give_Way | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Junction | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: No_Exit | Nulos/NaN: 0 | Porcentaje:

In [17]:
tipos = dict(weather.dtypes)
tipos

{'ID': 'string',
 'Weather_Timestamp': 'timestamp',
 'Temperature(F)': 'double',
 'Wind_Chill(F)': 'double',
 'Humidity(%)': 'double',
 'Pressure(in)': 'double',
 'Visibility(mi)': 'double',
 'Wind_Direction': 'string',
 'Wind_Speed(mph)': 'double',
 'Precipitation(in)': 'double',
 'Weather_Condition': 'string',
 'Amenity': 'boolean'}

In [44]:
weather = weather.dropna()

In [49]:

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType
exprs = []
total_filas = weather.count()
cols = [c for c in weather.columns if c != "ID"]
tipos = dict(weather.dtypes)
valores_nulos_texto = ["", "NaN", "nan", "None", "N/A", "NULL"]
for col_nombre in cols:
    if tipos[col_nombre] in ("double", "float"):
        condicion = F.col(col_nombre).isNull()| F.isnan(F.col(col_nombre))
    elif tipos[col_nombre] in ("timestamp", "boolean"):
        condicion = F.col(col_nombre).isNull()
    else: 
        condicion = F.trim(F.col(col_nombre)).isin(valores_nulos_texto) | F.col(col_nombre).isNull()
    exprs.append(F.sum(F.when(condicion, 1).otherwise(0)).alias(col_nombre))
conteos = weather.agg(*exprs).collect()[0].asDict()

In [50]:
for c in cols:
    n = conteos[c] or 0
    print(f"Columna: {c} | Nulos/NaN: {n} | Porcentaje: {n / total_filas * 100:.2f}%")

Columna: Weather_Timestamp | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Temperature(F) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Wind_Chill(F) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Humidity(%) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Pressure(in) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Visibility(mi) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Wind_Direction | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Wind_Speed(mph) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Precipitation(in) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Weather_Condition | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Amenity | Nulos/NaN: 0 | Porcentaje: 0.00%


In [68]:
temperaturas_bajas = weather.agg(F.min("Temperature(F)"))

In [69]:
temperaturas_bajas.show(100)

+-------------------+
|min(Temperature(F))|
+-------------------+
|              -45.0|
+-------------------+

